# PBG Fiber with an Air-Clad Outer Boundary

```{index} PBG; air outer region
```

The [first PBG notebook](4_1_pbg.ipynb) embeds
its lattice in cladding material that continues out to the 
PML (see the `lyr6cr2` configuration in `fiber_dicts`).
Real fibers are sometimes coated or suspended in air instead, so this
notebook reuses the *same* lattice via `fiber_dicts.lyr6cr2_w_air`,
whose only difference is that the outer/PML region is air
($n_\text{outer}=1$) rather than more cladding.

That single change breaks the search-center shortcut from the first
notebook's tip: `A.sqrZfrom(beta)` depends on the base index `n0`
used to nondimensionalize the eigenproblem, and here `n0`$=n_\text{outer}$
instead of `n0`$=n_\text{clad}$. The same physical mode therefore sits
at a *different* nondimensional $Z$ than it did in the first
notebook. 

In [ ]:
import ngsolve as ng
import numpy as np
from ngsolve.webgui import Draw
from fibermode import PBG
from fibermode.pbg.fiber_dicts.lyr6cr2_w_air import params

## Constructing the fiber

```{index} custom parameters; override fiber_dicts
```
```{index} fiber_dicts; override
```

In [ ]:
# Take the default parameters and triple the PML thickness

params2 = dict(params)
params2['r_out'] = params['r_pml'] + 3 * (params['r_out'] - params['r_pml'])  # triple the PML thickness
A = PBG(params2)

# Draw the refractive index profile

Draw(A.N, A.mesh, 'index', autoscale=False, min=0.95*A.n_clad, max=A.n_tube,
     settings={"Objects": {"Wireframe": False, "Edges":False}})
A.n_tube, A.n_clad, A.n_outer

## Finding a search center when $n_0 \neq n_\text{clad}$

```{index} search center; when n0 not clad
```
```{index} sqrZfrom; outer region mismatch
```

The recipe consists of  first finding the *physical* propagation constant $\beta$ of
the mode of interest using a fiber where the shortcut from the first
notebook applies. Here  the plain `lyr6cr2` fiber, whose outer region
matches its cladding. This is exactly the fundamental mode already
located in that notebook. Then,  convert that same $\beta$ into a
search center *for this fiber* by passing it through *this* object's
`A.sqrZfrom`, which nondimensionalizes using this fiber's own `n0`.
Because $n_\text{outer}=1$ is far from $n_\text{clad}$, the resulting
$Z$ lands in a new spot (here, on the imaginary axis)
rather than somewhere a naive guess would have looked.

In [ ]:
beta_clad = 5874506.48513697 + 1.67830311e-05j  # from the fundamental
                                                 # mode of `lyr6cr2`
                                                 # (see 4.1)
z_clad = A.sqrZfrom(beta_clad) ** .5  # sqrt: leakymode searches the
                                       # Z-plane, not Z^2
z_clad

## Scalar search

With the center in hand, a small-radius search around it should
capture the fundamental mode (along with, typically, a couple of
*cladding modes*, i.e., modes of the lattice itself rather than the
core defect, that also happen to fall within the same small contour).
These are usually easy to spot by eye: unlike the core-guided mode,
their field isn't concentrated in the central defect.

In [ ]:
z, y, yl, beta, P, extras = A.leakymode(
    3,
    rad=.001,
    ctr=z_clad,
    alpha=A.alpha,
    stop_tol=1e-8,
    niterations=5, npts=4,
    nspan=4, nrestarts=0,
)

In [ ]:
for f in y:
    Draw(f, A.mesh, settings={"Objects": {"Wireframe": False, "Edges": True}})

The results are a mix of  cladding modes and 
the core-guided fundamental mode clearly peaked in the
central defect.

## Vector search

As in the [rod-lattice notebook](4_2_pbg_rod.ipynb), we now center the vector search
at the square of the scalar result, modified  
to sit closer to the core-guided mode alone and away from the
cladding modes identified above.

In [ ]:
z**2

This motivates the `center2` value below.

In [ ]:
center2 = -1406.3

betas, Zsqrs, Es, phis, R = A.leakyvecmodes(
    p=1,
    rad=.1,
    ctr=center2,
    alpha=A.alpha,
    stop_tol=1e-8,
    quadrule='ellipse_trapez_shift',
    rhoinv=.9,
    niterations=20, npts=2,
    nspan=1, nrestarts=0,
)

In [ ]:
for e in Es:
    Draw(e.Norm(), A.mesh, settings={"Objects": {"Wireframe": False, "Edges": True}})
for phi in phis:
    Draw(phi, A.mesh, settings={"Objects": {"Wireframe": False, "Edges": True}})

## Summary

The recipe used here may generalize beyond for other fibers: whenever the
outer/PML region's index doesn't match the cladding's ($n_0 \neq
n_\text{clad}$), locate the physical $\beta$ of the mode of interest
in a matched-index version of the same lattice first, convert it
through *this* fiber's own `A.sqrZfrom`, expect a few spurious
cladding modes in the resulting small-radius scalar search, and refine
the vector search center away from them once identified.

```{index} holey fiber
```
```{index} PCF; index-guided example
```

Varying the outer region's index isn't the only way to get a
different-looking fiber out of the same `PBG` class: changing the
*lattice*'s index instead (making the "tube" sites lower-index than
the cladding rather than higher) turns this into an 
index-guided  photonic crystal fiber (PCF) in the narrower
sense.  See `fiber_dicts.holey` and
`demos/pbg/holey_demo.py` for that contrasting example, and the
[first notebook](4_1_pbg.ipynb)'s discussion of PBG vs. PCF.